# import_controls

> Import UI with file input, merge strategy selector, and result display

In [ ]:
#| default_exp components.import_controls

In [ ]:
#| export
from typing import Any

from fasthtml.common import Div, Span, H3, Form, Input, Button, P, Label

# DaisyUI components
from cjm_fasthtml_daisyui.components.actions.button import (
    btn, btn_colors, btn_sizes
)
from cjm_fasthtml_daisyui.components.data_input.file_input import file_input
from cjm_fasthtml_daisyui.components.data_input.radio import radio
from cjm_fasthtml_daisyui.components.data_input.fieldset import (
    fieldset, fieldset_legend, label as fieldset_label
)
from cjm_fasthtml_daisyui.components.feedback.alert import alert, alert_colors
from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui, bg_dui, border_dui
from cjm_fasthtml_daisyui.utilities.border_radius import border_radius

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, items, gap
)
from cjm_fasthtml_tailwind.utilities.borders import border
from cjm_fasthtml_tailwind.core.base import combine_classes

# Icons
from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Local imports
from cjm_transcript_workflow_management.models import ManagementUrls, ImportResult
from cjm_transcript_workflow_management.html_ids import ManagementHtmlIds
from cjm_transcript_workflow_management.components.helpers import DEBUG_MANAGEMENT_RENDER

## Merge Strategy Options

In [ ]:
#| export
MERGE_STRATEGIES = [
    ("skip", "Skip", "Skip nodes that already exist (default)"),
    ("overwrite", "Overwrite", "Replace existing nodes with imported data"),
    ("merge", "Merge", "Merge properties of matching nodes"),
]

## Import Result Display

In [ ]:
#| export
def render_import_result(
    result:ImportResult,  # Import operation result
) -> Any:  # Alert element showing import outcome
    """Render the import result as a success or error alert."""
    if result.success:
        parts = []
        if result.nodes_created or result.edges_created:
            parts.append(f"Imported: {result.nodes_created} nodes, {result.edges_created} edges")
        if result.nodes_skipped or result.edges_skipped:
            parts.append(f"Skipped: {result.nodes_skipped} nodes, {result.edges_skipped} edges")
        message = ". ".join(parts) if parts else "Import complete (no changes)."
        color = str(alert_colors.success)
    else:
        message = "; ".join(result.errors) if result.errors else "Import failed."
        color = str(alert_colors.error)
    
    return Div(
        Span(message),
        role="alert",
        cls=combine_classes(alert, color),
        id=ManagementHtmlIds.IMPORT_RESULT,
    )

In [ ]:
from fasthtml.common import to_xml
from cjm_transcript_workflow_management.models import ImportResult

# Success case
ok = render_import_result(ImportResult(
    success=True, nodes_created=248, edges_created=494,
    nodes_skipped=0, edges_skipped=0,
))
xml = to_xml(ok)
assert "248 nodes" in xml
assert "494 edges" in xml
assert "alert-success" in xml
print("Success result OK")

Success result OK


In [ ]:
# Error case
err = render_import_result(ImportResult(
    success=False, nodes_created=0, edges_created=0,
    nodes_skipped=0, edges_skipped=0,
    errors=["Invalid format: expected 'cjm-context-graph'"],
))
xml = to_xml(err)
assert "Invalid format" in xml
assert "alert-error" in xml
print("Error result OK")

Error result OK


In [ ]:
# Skip case
skip = render_import_result(ImportResult(
    success=True, nodes_created=0, edges_created=0,
    nodes_skipped=248, edges_skipped=494,
))
xml = to_xml(skip)
assert "Skipped: 248 nodes" in xml
assert "alert-success" in xml
print("Skip result OK")

Skip result OK


## Import Controls

In [ ]:
#| export
def render_import_controls(
    urls:ManagementUrls,  # URL bundle for route endpoints
) -> Any:  # Import form with file input, strategy selector, and result area
    """Render the import section with file input, merge strategy, and submit button."""
    if DEBUG_MANAGEMENT_RENDER:
        print("[RENDER] import_controls")
    
    # Merge strategy radio buttons
    strategy_radios = []
    for i, (value, label_text, description) in enumerate(MERGE_STRATEGIES):
        strategy_radios.append(
            Label(
                Input(
                    type="radio",
                    name="merge_strategy",
                    value=value,
                    cls=str(radio),
                    **(dict(checked=True) if i == 0 else {}),
                ),
                Div(
                    Span(label_text, cls=str(font_weight.medium)),
                    Span(
                        f" \u2014 {description}",
                        cls=combine_classes(font_size.sm, text_dui.base_content.opacity(60)),
                    ),
                ),
                cls=combine_classes(flex_display, items.center, gap(2)),
            )
        )
    
    return Div(
        # Header
        H3(
            lucide_icon("upload", size=5),
            "Import Graph",
            cls=combine_classes(
                font_size.lg, font_weight.semibold,
                flex_display, items.center, gap(2),
                m.b(4),
            ),
        ),
        Form(
            # File input
            Div(
                Label("File", cls=combine_classes(font_weight.medium, font_size.sm)),
                Input(
                    type="file",
                    name="file",
                    accept=".json",
                    cls=combine_classes(file_input, w.full),
                ),
                cls=combine_classes(flex_display, flex_direction.col, gap(1)),
            ),
            # Merge strategy
            Div(
                Label("Merge Strategy", cls=combine_classes(font_weight.medium, font_size.sm)),
                Div(
                    *strategy_radios,
                    cls=combine_classes(flex_display, flex_direction.col, gap(2)),
                ),
                cls=combine_classes(flex_display, flex_direction.col, gap(2), m.t(4)),
            ),
            # Submit
            Button(
                lucide_icon("upload", size=4),
                "Import",
                cls=combine_classes(
                    btn, btn_colors.primary, btn_sizes.sm,
                    flex_display, items.center, gap(1),
                    m.t(4),
                ),
                type="submit",
            ),
            hx_post=urls.import_graph,
            hx_target=f"#{ManagementHtmlIds.IMPORT_RESULT}",
            hx_swap="outerHTML",
            hx_encoding="multipart/form-data",
        ),
        # Result area (initially empty)
        Div(id=ManagementHtmlIds.IMPORT_RESULT, cls=str(m.t(4))),
        id=ManagementHtmlIds.IMPORT_SECTION,
        cls=combine_classes(
            p(4), border_radius.box,
            border(), border_dui.base_content.opacity(5),
            bg_dui.base_100,
        ),
    )

In [ ]:
from cjm_transcript_workflow_management.models import ManagementUrls

urls = ManagementUrls(
    management_page="/manage/documents/management_page",
    list_documents="/manage/documents/list_documents",
    document_detail="/manage/documents/document_detail",
    delete_document="/manage/documents/delete_document",
    delete_selected="/manage/documents/delete_selected",
    export_document="/manage/export/export_document",
    export_all="/manage/export/export_all",
    import_graph="/manage/import/import_graph",
)

controls = render_import_controls(urls)
xml = to_xml(controls)
assert ManagementHtmlIds.IMPORT_SECTION in xml
assert ManagementHtmlIds.IMPORT_RESULT in xml
assert 'type="file"' in xml
assert 'accept=".json"' in xml
assert 'name="merge_strategy"' in xml
assert 'value="skip"' in xml
assert 'value="overwrite"' in xml
assert 'value="merge"' in xml
assert 'checked' in xml
assert 'Import Graph' in xml
assert urls.import_graph in xml
assert 'multipart/form-data' in xml
print("Import controls OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()